# Walking nudge method: refine coefficients with small changes

We have a set of 100 good sets of coefficients from the genetic algorithm. We expect that they can be improved with slight adjustments to the coefficient values. In this notebook we find a way to try all sensible combinations of nudging the values of the coefficients.

The previous notebook showed that there are better solutions available by trying a few at random. It also showed that we can't simplify the problem by optimising for deprivation level and age band separately, as accuracy in both measures is needed to achieve a good overall fitness. We want to try every sensible combination rather than nudging values at random to make sure we don't miss any good options. To try every combination of nudges for all 25 coefficients there are 5^25 = 3x10^17 (three hundred quadrillion) options, and lots of those options are probably rubbish - for example, we expect that nudging all or most values upwards will worsen the fit overall, so there's no sense in calculating those options. We need to find a way to select only sensible options and so slash the number of combinations that we need to try.

In this notebook we find a way to try all of the sensible nudges to the coefficients. 

The motivation behind the method is:

+ assume that the starting values are pretty good fits to the data, so
+ increasing or decreasing the value of only one coefficient is bound to make the fitness worse, and
+ the fitness overall might stay at the same level if we simultaneously increase one coefficient (to increase total admissions) and decrease another (to decrease total admissions).

The best chance of a good result is to keep the total admissions from all age bands or all deprivation levels about the same, but to shift the admissions to different age bands or to different deprivation levels. This can be achieved by increasing one coefficient and decreasing another in the same row or the same column when the 25 coefficients are arranged in a 5x5 grid.

By sticking to this principle, we only need to check the nudges to all pairs of coefficients that share a row or a column. Then we can pick the change that results in the best fitness and discard the other worse options. Then we find another change to improve fitness, and so on until no more improvements can be found. 

__Inputs__

+ MSOA-level admission numbers, numbers of people in each age band.
+ Probability of stroke given age band, i.e. SSNAP-derived coefficients
+ Total number of admissions in each age band from SSNAP
+ 100 sets of "best" coefficients from 100 runs of the genetic algorithm

__Method__

1. Set up the nudges available:
    + Make a list of the 200 pairs of coefficients. A pair cannot contain the same coefficient twice. The two orders are counted separately, i.e. coefficients A and B are paired up once as (A, B) and once as (B, A). The two paired coefficients must be in either the same row or the same column of the 5x5 grid.
    + Find all combinations of nudges.
    + Firstly, apply two nudges at once. Apply one offset to the first in the pair and a second offset to the second in the pair. The pairs of offsets are: (1, -1), (-2, 1), (2, -1), (2, -2).
    + Then, apply only one nudge at a time. Only one coefficient changes its value. The offsets available are: +1, -1, +2, -2.
    + This gives 900 sets of nudges (4 pairs of offsets with 200 combinations and 4 individual offsets with 25 combinations: 4x200 + 4x25).
    + Multiply the nudge value used by the significant value required. Anything for the under 65 age band uses 1e-4 as its significant figure, and all other age bands use 1e-3.
1. Apply the nudges to a set of coefficients.
    +  Add or subtract the one or two significant figures as required. This makes 900 sets of nudged coefficients.
    +  Check whether each set of coefficients still increases with age band and with deprivation level. If it fails either test, discard the set of coefficients.
5. Calculate the fitnesses of the remaining sets of nudged coefficients.
6. Keep the set with the best fitness. Discard the rest.
7. Return to step 2 _unless_ the same set of best coefficients has been picked twice, in which case there are no more improvements to be made.

This means that in each pass, we try a number of combinations that are quite close to the starting values. We prioritise changes that result in better fitness than the starting point, discarding changes that are worse, and iteratively make changes to find the best combination of coefficients near the start values.

We apply this walking nudge method to each of the 100 sets of coefficients from the runs of the genetic algorithm, and so find the best set of coefficients that is near to those starting points.

__Results__

In the next notebook.


In [1]:
import polars as pl
import os
import numpy as np
import itertools

# Print general numbers, not scientific notation:
np.set_printoptions(suppress=True)

## Load data

### Admissions

In [2]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [3]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [4]:
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}'
               for q in qmin_list for a in age_numbers]
# Check the first few:
print(coeff_names[:3])

# Quantile names:
quantile_str_list = sorted(list(set([c.split('_')[-1] for c in coeff_names])))

['age_less65_q00', 'age_65_q00', 'age_70_q00']


Gather admissions data by deprivation quantile:

In [5]:
admissions_lists = []
x_lists = []

for qmin in qmin_list:
    df_stats_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)
    # MSOA data in the same order as those coefficients:
    x_lists_here = [df_stats_here[a] for a in age_numbers]
    admissions_here = df_stats_here['admissions'].to_numpy().tolist()
    # Store:
    admissions_lists.append(admissions_here)
    x_lists.append(x_lists_here)

### Age-admissions coefficients

Starting SSNAP coefficients:

In [6]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [7]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [8]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Turn them into a dictionary:

In [9]:
labels = ['less65', '65', '70', '75', 'over80']
coeffs_ssnap_dict = (
    dict(zip(labels, df_pop_admissions['prob_stroke_given_age'].to_numpy())))

coeffs_ssnap_dict

{'less65': 0.000408,
 '65': 0.002644,
 '70': 0.003735,
 '75': 0.006005,
 'over80': 0.011558}

Pick out admissions numbers:

In [10]:
admissions_by_age = (
    df_pop_admissions['admissions_annual_boost'].to_numpy().flatten())

### Best results from genetic algorithm

Data stored as scale factors for the SSNAP-derived coefficients.

In [11]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [12]:
df_best_gens.head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,1.3,1.2,1.332,1.237,1.2,1.026,1.145,1.151,1.041,1.1,1.0,1.0,0.988,0.938,1.0,0.862,0.9,0.945,0.938,0.937,0.8,0.9,0.732,0.832,0.874,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,1.1,1.3,1.305,1.3,1.231,1.1,1.1,1.1,1.023,1.1,1.0,1.0,1.0,1.0,1.0,0.969,0.9,0.9,0.892,0.9,0.837,0.849,0.8,0.8,0.894,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,1.273,1.308,1.175,1.2,1.214,0.981,1.1,1.103,1.1,1.131,0.949,0.987,1.046,1.0,1.0,0.949,0.979,0.896,0.9,0.9,0.844,0.773,0.861,0.8,0.88,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,1.171,1.3,1.327,1.186,1.3,1.1,1.198,1.1,1.1,1.1,1.042,1.0,1.0,1.0,0.971,0.881,0.9,0.894,1.0,0.9,0.8,0.771,0.808,0.7,0.9,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,1.2,1.2,1.25,1.225,1.293,1.1,1.1,1.11,1.069,1.1,0.976,1.019,1.0,1.0,0.961,0.925,0.904,0.9,0.915,0.91,0.8,0.9,0.844,0.8,0.879,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993


Convert the scale factors to the actual age-deprivation coefficient values using the SSNAP coefficients.

In [13]:
for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = df_best_gens[coeff] * coeffs_ssnap_dict[key]
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Round results:

In [14]:
# Set up dictionary with how many decimal places to round to:
labels = ['less65', '65', '70', '75', 'over80']
round_dict = dict(zip(labels, [4, 3, 3, 3, 3]))
# round_dict = dict(zip(labels, [5, 4, 4, 4, 3]))

for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = np.round(df_best_gens[coeff], round_dict[key])
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Drop the measures of goodness of fit now that the coefficients have been rounded:

In [15]:
df_best_gens = df_best_gens.drop(
    ['fitness'] + [c for c in df_best_gens.columns if c.startswith('r2')])

## Recalculate fitness

Use similar functions to previous notebooks to calculate sum of square residuals and fitness.

In [16]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat.to_numpy().tolist()

In [17]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [18]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (np.array(yhat) - np.array(y))**2.0
    # Sum of differences:
    sum_sqres = sqres.sum()
    return np.sqrt(sum_sqres)

Goodness check option 3: This calculates the sum of the ratio of the error to each observed number of strokes:

In [19]:
# def find_err_ratios(yhat, y):
#     # Difference from actual:
#     sqres = np.abs(np.array(yhat) - np.array(y)) / np.array(y)
#     # Sum of differences:
#     sum_sqres = sqres.sum()
#     return sum_sqres

In [20]:
def many_sum_sqres(individual, x_lists, admissions_lists):
    # Predictions for each MSOA:
    predictions_lists = []
    sum_sqres_by_depriv = []
    for i in range(5):
        coeffs = individual[(i*5):(i*5)+5]  # coeffs for this depriv.
        yhat = predict_admissions(x_lists[i], coeffs)
        sum_sqres = find_square_residuals(yhat, admissions_lists[i])
        # sum_sqres = find_err_ratios(yhat, admissions_lists[i])
        predictions_lists.append(yhat)
        sum_sqres_by_depriv.append(sum_sqres)


    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Calculate sum of square residuals across all MSOA:
    sum_sqres = find_square_residuals(predicted_all, observed_all)
    # sum_sqres = find_err_ratios(predicted_all, observed_all)

    return sum_sqres, sum_sqres_by_depriv

In [21]:
# the goal ('fitness') function to be maximized
def eval_admissions(individual, admissions_lists, x_lists, admissions_by_age):

    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0

    sum_sqres, sum_sqres_by_depriv = many_sum_sqres(individual, x_lists, admissions_lists)
    
    return (sum_sqres, rat, sum_sqres * rat, predictions_by_age, sum_sqres_by_depriv)

In [22]:
def eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=False):
    eval_dict = {}
    eval_labels = [
        'sum_sqres',
        'wrong_rat',
        'fitness',
        'wrong_rat_under65',
        'wrong_rat_65',
        'wrong_rat_70',
        'wrong_rat_75',
        'wrong_rat_over80',
        'sum_sqres_q00',
        'sum_sqres_q02',
        'sum_sqres_q04',
        'sum_sqres_q06',
        'sum_sqres_q08',
    ]
    eval_dict = dict(zip(eval_labels, [[] for e in eval_labels]))
    
    for d in range(len(df_best_gens)):
        df = df_best_gens[d]
        # Pick out coefficients:
        coeffs = df[coeff_names].to_numpy().flatten()
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs,
            admissions_lists,
            x_lists,
            admissions_by_age
        )
        eval_dict['sum_sqres'].append(sum_sqres)
        eval_dict['wrong_rat'].append(rat)
        eval_dict['fitness'].append(fitness)
        eval_dict['wrong_rat_under65'].append(ratios_by_age[0])
        eval_dict['wrong_rat_65'].append(ratios_by_age[1])
        eval_dict['wrong_rat_70'].append(ratios_by_age[2])
        eval_dict['wrong_rat_75'].append(ratios_by_age[3])
        eval_dict['wrong_rat_over80'].append(ratios_by_age[4])
        eval_dict['sum_sqres_q00'].append(sum_sqres_by_depriv[0])
        eval_dict['sum_sqres_q02'].append(sum_sqres_by_depriv[1])
        eval_dict['sum_sqres_q04'].append(sum_sqres_by_depriv[2])
        eval_dict['sum_sqres_q06'].append(sum_sqres_by_depriv[3])
        eval_dict['sum_sqres_q08'].append(sum_sqres_by_depriv[4])

    # Either start a new blank dataframe or add the fitnesses to the
    # input df:
    df_new = df_best_gens if concat else pl.DataFrame()
    for key, v in eval_dict.items():
        df_new = df_new.with_columns(pl.Series(key, v))
    return df_new

Recalculate fitnesses for the genetic algorithm outputs:

In [23]:
df_best_gens = eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=True)

View results:

In [24]:
df_best_gens.sort('fitness').head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed74""",22.0,0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696
"""randomseed10""",26.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.357618,1.052517,252.980542,0.980725,1.01735,0.990024,0.99959,0.994494,110.501962,107.849722,108.96863,107.721177,102.232696
"""randomseed94""",18.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.67206,1.061548,253.361768,0.980725,1.01735,0.990024,0.99959,1.014536,110.501962,107.849722,108.96863,103.905822,102.232696
"""randomseed26""",20.0,0.0005,0.004,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.014,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.532221,1.061076,254.161938,0.980725,1.01735,0.990024,0.994136,0.991389,110.922311,107.800573,106.750111,107.721177,102.232696
"""randomseed57""",56.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.005,0.01,239.371525,1.062211,254.262948,0.980725,1.01735,0.990024,1.010103,0.994494,110.501962,108.40745,108.96863,104.929413,102.232696


Pick out the best coeffs so far:

In [25]:
df_coeffs_best_deap = df_best_gens.sort('fitness')[0]
coeffs_best_deap = df_coeffs_best_deap[coeff_names].to_numpy().flatten()

## Check effect of nudges

When one of the coefficients is changed by one significant figure, what is the effect on the admissions numbers?

Calculate the number of people in each combination of deprivation and age band:

In [26]:
dict_total_persons = {}

for i, lists_by_depriv_band in enumerate(x_lists):
    q_here = quantile_str_list[i]
    for lists_by_age in lists_by_depriv_band:
        age_band = lists_by_age.name
        n_people = lists_by_age.sum()
        dict_total_persons[f'{age_band}_{q_here}'] = n_people

In [27]:
arr_total_persons = np.array(list(dict_total_persons.values())).reshape(5, 5)

arr_total_persons

array([[9924432.9305    ,  454870.6392    ,  408533.6541    ,
         282735.5075    ,  404335.3945    ],
       [9602442.0019    ,  508547.6151    ,  490250.9899    ,
         346299.4298    ,  492042.4994    ],
       [9004763.90820001,  582915.6219    ,  595289.7078    ,
         423567.5196    ,  593857.2014    ],
       [8520385.1629    ,  615871.6872    ,  650064.9713    ,
         468822.7434    ,  657198.8848    ],
       [8544149.7571    ,  598979.519     ,  647016.1588    ,
         472220.0104    ,  685473.2984    ]])

One significant figure for each coefficient:

In [28]:
round_nudge = np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3] * 5).reshape(5, 5)
# round_nudge = np.array([1e-5, 1e-4, 1e-4, 1e-4, 1e-3] * 5).reshape(5, 5)

Multiply these by the total numbers of people to find the effect of one significant figure on the number of admissions:

In [29]:
np.round(arr_total_persons * round_nudge, 1)

array([[992.4, 454.9, 408.5, 282.7, 404.3],
       [960.2, 508.5, 490.3, 346.3, 492. ],
       [900.5, 582.9, 595.3, 423.6, 593.9],
       [852. , 615.9, 650.1, 468.8, 657.2],
       [854.4, 599. , 647. , 472.2, 685.5]])

Adding or subtracting one click from each coefficient has an effect on the order of half a thousand admissions (0.6% of England total) for all coeffs except the under-65 age band where the effect is nearer one thousand admissions (1.3% of England total).

Test the effect on a few MSOA at random:

In [30]:
np.random.seed(42)

msoa_names = np.random.choice(df_stats['MSOA'], size=5, replace=False)

msoa_names

array(['Lancaster 016', 'Hammersmith and Fulham 012',
       'Aylesbury Vale 017', 'Havering 030', 'Fareham 008'], dtype='<U32')

In [31]:
labels = ['MSOA', 'admissions'] + [f'nudge_{a}' for a in age_numbers] 
cols = [[] for i in range(len(labels))]

for msoa in msoa_names:
    df = df_stats.filter(df_stats['MSOA'] == msoa)
    admissions_here = df['admissions'][0]
    x_lists_here = [df[a][0] for a in age_numbers]
    admissions_nudge = np.array(x_lists_here) * round_nudge.flatten()[:5]
    cols[0].append(msoa)
    cols[1].append(np.round(admissions_here, 1))
    for i, ad in enumerate(admissions_nudge):
        cols[i+2].append(np.round(ad, 1))

df_msoa_test = pl.DataFrame()
for i, label in enumerate(labels):
    df_msoa_test = df_msoa_test.with_columns(pl.Series(label, cols[i]))

In [32]:
df_msoa_test

MSOA,admissions,nudge_age_less65,nudge_age_65,nudge_age_70,nudge_age_75,nudge_age_over80
str,f64,f64,f64,f64,f64,f64
"""Lancaster 016""",31.0,1.1,0.8,0.8,0.6,0.7
"""Hammersmith and Fulham 012""",7.7,0.8,0.3,0.4,0.2,0.4
"""Aylesbury Vale 017""",13.0,0.7,0.5,0.5,0.4,0.7
"""Havering 030""",6.3,0.6,0.3,0.2,0.2,0.3
"""Fareham 008""",16.7,0.8,0.5,0.6,0.5,0.7


The effect of nudging the values is harder to describe on the MSOA level because of the different ratios of people in each age band. For some of the above, the nudge has twice as much effect in the under-65 column. For some of the above, the nudge has a somewhat larger effect on the over-80 column than the middle three age bands. The size of a nudge is typically under 5% of the MSOA admissions but is around 10% in some cases where the total admissions is low.

The effect of nudging the values by one significant figure is small enough that we can justify using only one significant figure instead of two, which would mean exploring a lot more options and have more potential of falling into a local minimum.

Some of the coefficients potentially have twice the effect on total admissions as others. Therefore when deciding how to nudge coefficients, we should include the option of moving one coefficient by one click and another by two clicks.

## Set up pairs of coefficients

Set up coefficient locations/indices in the format (row number, column number):

In [33]:
row_col_inds = [c for c in itertools.product(range(5), repeat=2)]

row_col_inds

[(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (1, 0),
 (1, 1),
 (1, 2),
 (1, 3),
 (1, 4),
 (2, 0),
 (2, 1),
 (2, 2),
 (2, 3),
 (2, 4),
 (3, 0),
 (3, 1),
 (3, 2),
 (3, 3),
 (3, 4),
 (4, 0),
 (4, 1),
 (4, 2),
 (4, 3),
 (4, 4)]

The following function finds all pairs of coefficients that share a row or column:

In [34]:
def build_pairs(row_col_inds):
    """
    Find a list of pairs of coefficient indices.

    Excludes pairs with the same coefficient twice.

    Inputs
    ------
    row_col_inds - list. List of tuple/list, one pair of numbers for
                   each coefficient's row then column index.

    Returns
    -------
    pairs_list - list. Each item is a tuple of two tuples, and each
                 of the inner tuples is a row then column index for
                 a coefficient. The two outer pairs of tuples are
                 for the two coefficients that have been paired up.
    """
    pairs_list = []
    for r, (row_a, col_a) in enumerate(row_col_inds):
        for col_b in range(5):
            # Compare with each other point in the same row:
            if col_b != col_a:
                pairs_list.append(((row_a, col_a), (row_a, col_b)))
        for row_b in range(5):
            # Compare with each other point in the same column:
            if row_b != row_a:
                pairs_list.append(((row_a, col_a), (row_b, col_a)))
    return pairs_list

Find all 200 combos of pairs of indices in the same row/column:

In [35]:
row_col_pairs = build_pairs(row_col_inds)

Check how many pairs there are. Expected result is 8 pairs for each first coefficient in the pair, and 25 first coefficients, for 8 x 25 = 200 pairs.

In [36]:
len(row_col_pairs)

200

Check the first few pairs and also extracting rows and columns from the lists:

In [37]:
for r, ((row_a, col_a), (row_b, col_b)) in enumerate(row_col_pairs[:10]):
    print(r, row_a, col_a, row_b, col_b)

0 0 0 0 1
1 0 0 0 2
2 0 0 0 3
3 0 0 0 4
4 0 0 1 0
5 0 0 2 0
6 0 0 3 0
7 0 0 4 0
8 0 1 0 0
9 0 1 0 2


Convert (row, col) indices to position in the flat list of 25 values:

In [38]:
flat_inds_pairs = []

for r, ((row_a, col_a), (row_b, col_b)) in enumerate(row_col_pairs):
    flat_inds_pairs.append([row_a*5 + col_a, row_b*5 + col_b])

flat_inds_arr = np.hstack((np.arange(len(flat_inds_pairs)).reshape(len(flat_inds_pairs), 1), np.array(flat_inds_pairs)))

Calculate all combinations of nudges to a single set of coefficients.

In [39]:
# First make a copy of zero offsets for as many pairs of indices as there are:
nudge_arr = np.zeros((len(flat_inds_pairs), 25))
# Update the values:
# First index in each pair set to +1:
nudge_arr[flat_inds_arr[:, 0], flat_inds_arr[:, 1]] += 1
# Second index in each pair set to 2:
nudge_arr[flat_inds_arr[:, 0], flat_inds_arr[:, 2]] += 2

Check the first few options:

In [40]:
nudge_arr[:3]

array([[1., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.]])

## Nudge values

Set up the walking nudge method. Start with some set of coefficients that is near the best answer. Then calculate new coefficients from nudging all pairs of coefficients that share a row or column. Pick the nudged set that has the best fitness, and then repeat calculating new nudged coefficients and picking the best. Keep iterating until no more improvements can be made.

The following function takes each pair of values, applies some offset, checks if the result obeys the rules (increase with age and with deprivation), and if so then keeps the result.

In [41]:
def build_nudge_df(nudge_arr, start_coeffs, coeff_names, step_dicts=None):
    """
    Nudge pairs of start coeffs to find new valid sets of coeffs.

    The pairs of coefficients that share a row or column are nudged
    so that the first one increases and the second decreases by
    one significant figure (200 combinations).
    Separately, each individual coefficient is nudged up (25 options)
    or downwards (25 options) by one significant figure.

    Inputs
    ------
    row_col_pairs - list. Each item is a tuple of two pairs of indices,
                    one for first coeff and one for second.
    start_coeffs  - list. 25 coefficients before nudging.
    coeff_names   - list. Column names for start_coeffs for conversion
                    to DataFrame.

    Returns
    -------
    df_nudges - pl.DataFrame. All valid nudged coefficients.
    """
    # Function to save repeating some code:
    def convert_to_dataframe(coeffs_here, coeff_names):
        """
        Convert list of coefficients to a DataFrame.

        Inputs
        ------
        coeffs_here - list. Coefficients.
        coeff_names - list. Column names in same order as coeffs_here.
    
        Returns
        -------
        df_here - pl.DataFrame. DataFrame of coefficients with one row
                  and a column for each coefficient.
        """
        # Convert to dataframe:
        data_here = coeffs_here.flatten()
        df_here = pl.DataFrame(np.array(data_here).reshape(1, len(data_here)), schema=coeff_names)
        return df_here

    # Store results in here:
    df_nudges = pl.DataFrame()
    
    # Set up the changes applied to each pair of coefficients:
    # the first in the pair increases and the second decreases.
    # n.b. don't need to run both combinations (1, -1) and (-1, 1)
    # because the big list of pairs of indices contains both
    # pairs (A, B) and (B, A).
    m_pairs_available = [
        (1.0, -1.0), (2.0, -1.0), (-2.0, 1.0), (2.0, -2.0),
        (2.0,), (-2.0,), (1.0,), (-1.0,)
    ]

    if step_dicts is None:
        # Set up which step sizes are used in which columns:                             REMEMBER TO CHANGE THIS!!!! -----------------------------------------------
        step_dicts = {
            'age_less65': {'step': 1e-4, 'inds': [0, 5, 10, 15, 20]},
            'age_over65': {'step': 1e-3, 'inds': []},
            # 'age_less65': {'step': 1e-5, 'inds': [0, 5, 10, 15, 20]},
            # 'age_over80': {'step': 1e-3, 'inds': [4, 9, 14, 19, 24]},
            # 'age_middle': {'step': 1e-4, 'inds': []},
            # 'age_less65': {'step': 1e-5, 'inds': [0, 5, 10, 15, 20]},
            # 'age_over65': {'step': 1e-4, 'inds': []},
        }
        # Let the other option have all other inds:
        step_dicts['age_over65']['inds'] = list(set(np.arange(25)) - set(step_dicts['age_less65']['inds']))
        # step_dicts['age_middle']['inds'] = list(set(np.arange(25)) - (
        #     set(step_dicts['age_less65']['inds']) | set(step_dicts['age_over80']['inds'])))
    for m_pairs in m_pairs_available:
        # Start with one copy of the start coeffs for each row in nudge_arr:
        arr_nudged_coeffs = np.tile(start_coeffs, (len(nudge_arr), 1))
        # Set up offsets to be added in here:
        offsets_grid = np.zeros((len(nudge_arr), 25))
        # Calculate which changes happen where:
        for dict_label, step_dict in step_dicts.items():
            # Only allow changes in the given columns:
            col_bool = np.zeros((len(nudge_arr), 25))
            col_bool[:, step_dict['inds']] = 1
            for m, m_change in enumerate(m_pairs):
                # Find where changes were made:
                change_bool = (nudge_arr == m+1)
                # Keep only where changes are allowed:
                mask = change_bool & col_bool.astype(bool)
                # Calculate changes:
                offsets_grid[mask] += m_change * step_dict['step']
        # Apply changes:
        arr_nudged_coeffs += offsets_grid
        # Convert to dataframe and stack onto existing:
        df = pl.DataFrame(arr_nudged_coeffs, schema=coeff_names)
        df_nudges = pl.concat((df_nudges, df))
   
    # Remove repeats:
    df_nudges = df_nudges.unique()
    return df_nudges

The following function removes invalid sets of coefficients from the nudged dataframe:

In [42]:
def remove_invalid_from_nudge_df(df_nudges, coeff_names):
    # Set up each set of columns that should be compared:
    age_cols_sets = [
        np.arange(i, i+5, 1) for i in np.arange(0, 25, 5)
    ]
    depriv_cols_sets = [
        np.arange(i, 25, 5)[::-1] for i in np.arange(0, 5, 1)
    ]
    # Reverse order of depriv cols so can check for increasing
    # diff rather than decreasing.

    coeffs_arr = df_nudges[coeff_names].to_numpy()
    mask_valid = np.ones(len(df_nudges))
    for cols in age_cols_sets + depriv_cols_sets:
        mask = np.all(np.diff(coeffs_arr[:, cols]) >= 0, axis=1)
        mask_valid *= mask.astype(int)
    # Only keep sets of coeffs that passed all tests:
    coeffs_arr = coeffs_arr[mask_valid.astype(bool)]
    # Convert back to dataframe:
    df_nudges_valid = pl.DataFrame(coeffs_arr, schema=coeff_names)
    return df_nudges_valid    

The following function calculates the fitnesses of each set of nudged values in turn and returns the results in a DataFrame:

In [43]:
def find_nudged_fitnesses(df_nudges):
    """
    Calculate the fitness of each set of nudged coeffs in turn.
    
    Fitnesses are only stored if the nudged coeffs increase with age
    band and with deprivation level. Otherwise NaN is stored.

    Inputs
    ------
    df_nudges - pl.DataFrame. One row per set of nudged coeffs.

    Returns
    -------
    df_nudges_fitness - pl.DataFrame(). A copy of the input DataFrame
                        with fitness metrics included.
    """
    df_nudges_fitness = pl.DataFrame()

    # Column names for resulting DataFrame:
    fitness_keys = [
        'sum_sqres', 'wrong_rat', 'fitness', 'wrong_rat_under65',
        'wrong_rat_65', 'wrong_rat_70', 'wrong_rat_75', 'wrong_rat_over80',
        'sum_sqres_q00', 'sum_sqres_q02', 'sum_sqres_q04', 'sum_sqres_q06',
        'sum_sqres_q08',
    ]
    for i in range(len(df_nudges)):
        # Pick out coeffs:
        coeffs = df_nudges[i][coeff_names].to_numpy().flatten()
        # If coeffs are valid, then calculate fitness:
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs,
            admissions_lists,
            x_lists,
            admissions_by_age
        )

        # Gather values for results:
        results_values = np.concatenate((coeffs.flatten(), [np.round(sum_sqres, 5), np.round(rat, 5), np.round(fitness, 5)], [np.round(r, 5) for r in ratios_by_age], [np.round(s, 5) for s in sum_sqres_by_depriv]))
        df_coeffs = pl.DataFrame(results_values.reshape(1, len(results_values)), schema=coeff_names + fitness_keys)
        # Store result:
        df_nudges_fitness = pl.concat((df_nudges_fitness, df_coeffs))
    return df_nudges_fitness

### Test run on a copy of the starting SSNAP-derived coefficients

Make a test set of coefficients by making five copies of the SSNAP coefficients, rounded to one significant figure:

In [44]:
coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [45]:
coeffs_ssnap_round = [0.0004, 0.003, 0.004, 0.006, 0.012]
# coeffs_ssnap_round = [0.00041, 0.0026, 0.0037, 0.0060, 0.012]
# coeffs_ssnap_round = [0.00041, 0.0026, 0.0037, 0.0060, 0.0116]

In [46]:
start_coeffs = np.concatenate([coeffs_ssnap_round] * 5)

Run the two functions above to nudge the values and keep the best option.

Keep a copy of the best (lowest "fitness" score) set of coefficients from each iteration of the nudging.

The nudging continues until the same set of coefficients have been chosen twice, and we'd expect any continued nudging to flick back and forth between two good sets of values. The nudging is cut off early if it takes more than 100 sets of nudges to settle on an answer.

In [47]:
start_coeffs = np.concatenate([coeffs_ssnap_round] * 5)
base_coeffs = np.array([c for c in start_coeffs])

df_nudge_history = find_nudged_fitnesses(pl.DataFrame(base_coeffs.reshape(1, len(base_coeffs)), schema=coeff_names))

keep_going_please = True
k = 0
while keep_going_please:
    df_nudges = build_nudge_df(nudge_arr, base_coeffs, coeff_names)
    df_nudges = remove_invalid_from_nudge_df(df_nudges, coeff_names)
    df_nudges = find_nudged_fitnesses(df_nudges)
    # Pick out the best one:
    df_nudges = df_nudges.sort('fitness')[0]
    base_coeffs = df_nudges[coeff_names].to_numpy().flatten()
    df_nudge_history = pl.concat((df_nudge_history, df_nudges))
    k += 1
    if (k > 2) & (len(df_nudge_history) != len(df_nudge_history.unique())):
        keep_going_please = False
    elif k > 100:
        keep_going_please = False
        print('Cutting off after 100 iterations.')

View the change in fitness after each pass:

In [48]:
with pl.Config(set_tbl_rows=len(df_nudge_history)):
    display(df_nudge_history[['sum_sqres', 'wrong_rat', 'fitness']])

sum_sqres,wrong_rat,fitness
f64,f64,f64
274.67555,1.28547,353.08646
261.73422,1.18499,310.15132
253.36321,1.11904,283.52357
247.29731,1.10154,272.4085
245.67552,1.07206,263.37998
242.1509,1.06529,257.96127
243.0604,1.04903,254.97841
241.3718,1.03934,250.86733
238.63314,1.03724,247.51954


Best fitness achieved:

In [49]:
print(f"Starting fitness (this set):      {df_nudge_history['fitness'][0]:.2f}")
print(f"Best fitness (this set):          {df_nudge_history['fitness'].min():.2f}")
print('')
print(f"Best fitness (all starting sets): {df_best_gens['fitness'].min():.2f}")

Starting fitness (this set):      353.09
Best fitness (this set):          245.57

Best fitness (all starting sets): 252.55


Pretty good fitness straight off. The derived coefficients have comparable fitness with anything in the genetic algorithm outputs.

View the change in coefficients after each pass:

In [50]:
with pl.Config(set_tbl_rows=len(df_nudge_history)):
    display(df_nudge_history[coeff_names])

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012
0.0004,0.004,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.001,0.004,0.006,0.012
0.0004,0.004,0.005,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.001,0.002,0.006,0.012
0.0004,0.005,0.005,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.001,0.002,0.006,0.012
0.0004,0.005,0.005,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.001,0.002,0.006,0.01
0.0004,0.005,0.005,0.006,0.014,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0004,0.001,0.002,0.006,0.01
0.0004,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0004,0.001,0.002,0.005,0.01
0.0004,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.011,0.0004,0.001,0.002,0.005,0.01
0.0004,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.011,0.0004,0.002,0.002,0.005,0.01


View the change in coefficients from the starting values after each pass:

In [51]:
with pl.Config(set_tbl_rows=len(df_nudge_history)):
    display(pl.DataFrame(df_nudge_history[coeff_names].to_numpy() - np.array(start_coeffs), schema=coeff_names))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,0.0,0.0,0.0
0.0,0.001,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,-0.002,0.0,0.0
0.0,0.002,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001,0.0,0.0,0.0,0.0,-0.002,-0.002,0.0,0.0
0.0,0.002,0.001,0.0,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001,0.0,0.0,0.0,0.0,-0.002,-0.002,0.0,-0.002
0.0,0.002,0.001,0.0,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001,0.0,0.0,-0.001,0.0,-0.002,-0.002,0.0,-0.002
0.0,0.002,0.001,0.001,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001,0.0,0.0,-0.001,0.0,-0.002,-0.002,-0.001,-0.002
0.0,0.002,0.001,0.001,0.002,0.0,0.0,0.0,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.001,0.0,-0.001,-0.001,0.0,-0.002,-0.002,-0.001,-0.002
0.0,0.002,0.001,0.001,0.002,0.0,0.0,0.0,0.001,0.0,0.0,-0.001,0.0,0.0,0.0,0.0,-0.001,0.0,-0.001,-0.001,0.0,-0.001,-0.002,-0.001,-0.002


View the final derived coefficients:

In [52]:
start_coeffs.reshape(5, 5)

array([[0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ]])

In [53]:
df_best_test = df_nudge_history.filter(df_nudge_history['fitness'] == df_nudge_history['fitness'].min())[0]

df_best_test[coeff_names].to_numpy().flatten().reshape(5, 5)

array([[0.0004, 0.005 , 0.007 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.005 , 0.007 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.005 , 0.011 ],
       [0.0004, 0.002 , 0.002 , 0.005 , 0.01  ]])

### Main run on genetic algorithm outputs

Run these functions on each of the 100 sets of coefficients from the genetic algorithm output.

Keep a copy of the best (lowest "fitness" score) set of coefficients from each iteration of the nudging.

The nudging continues until the same set of coefficients have been chosen twice, and we'd expect any continued nudging to flick back and forth between two good sets of values. The nudging is cut off early if it takes more than 100 sets of nudges to settle on an answer.

In [54]:
dict_nudged_results = {}

for d, dir_name in enumerate(df_best_gens['dir']):
    print(f'{d+1} out of {len(df_best_gens)}', end='\r')
    start_coeffs = df_best_gens.filter(df_best_gens['dir'] == dir_name)[coeff_names].to_numpy().flatten()
    df_nudge_history = find_nudged_fitnesses(pl.DataFrame(start_coeffs.reshape(1, len(start_coeffs)), schema=coeff_names))
    
    keep_going_please = True
    k = 0
    while keep_going_please:
        df_nudges = build_nudge_df(nudge_arr, start_coeffs, coeff_names)
        df_nudges = remove_invalid_from_nudge_df(df_nudges, coeff_names)
        df_nudges = find_nudged_fitnesses(df_nudges)
        # Pick out the best one:
        df_nudges = df_nudges.sort('fitness')[0]
        start_coeffs = df_nudges[coeff_names].to_numpy().flatten()
        df_nudge_history = pl.concat((df_nudge_history, df_nudges))
        k += 1
        if (k > 2) & (len(df_nudge_history) != len(df_nudge_history.unique())):
            keep_going_please = False
        elif k > 100:
            keep_going_please = False
            print(f'\nCutting off {dir_name} after 100 iterations.\n')

    # Store results:
    dict_nudged_results[dir_name] = df_nudge_history

In [55]:
df_nudge_history

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_err_rats,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_err_rats_q00,sum_err_rats_q02,sum_err_rats_q04,sum_err_rats_q06,sum_err_rats_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1497.79502,1.19387,1788.17287,0.93525,0.95584,0.95066,0.96988,0.99449,297.0475,298.31448,334.73368,273.35103,294.34834
0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1500.48467,1.1484,1723.15398,0.98072,0.95584,0.95066,0.96988,0.99449,297.0475,298.31448,334.73368,276.04068,294.34834
0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1505.33949,1.10904,1669.48031,0.98072,0.95584,0.99002,0.96988,0.99449,301.90232,298.31448,334.73368,276.04068,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1506.08865,1.06492,1603.85828,0.98072,1.00004,0.99002,0.96988,0.99449,322.66756,298.31448,314.7176,276.04068,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1512.36229,1.0352,1565.6017,0.98072,1.00004,0.99002,0.99959,0.99449,322.66756,304.58812,314.7176,276.04068,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1505.50142,1.03327,1555.59114,0.98072,1.00004,0.99002,0.99959,0.99643,322.66756,304.58812,301.73257,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1510.42152,1.03017,1555.98512,0.98072,1.00004,0.99002,0.99959,0.99953,322.66756,296.52318,314.7176,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1505.50142,1.03327,1555.59114,0.98072,1.00004,0.99002,0.99959,0.99643,322.66756,304.58812,301.73257,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1510.42152,1.03017,1555.98512,0.98072,1.00004,0.99002,0.99959,0.99953,322.66756,296.52318,314.7176,282.16484,294.34834


Check the nudge history of one of the sets of coefficients:

In [56]:
dict_nudged_results['randomseed32']

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_err_rats,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_err_rats_q00,sum_err_rats_q02,sum_err_rats_q04,sum_err_rats_q06,sum_err_rats_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.003,0.004,0.006,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1498.78526,1.17266,1757.56214,0.98072,0.95584,0.95066,0.94562,0.99449,295.34809,298.31448,334.73368,276.04068,294.34834
0.0005,0.003,0.004,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1503.79876,1.12414,1690.4785,0.98072,0.95584,0.95066,0.99414,0.99449,300.36159,298.31448,334.73368,276.04068,294.34834
0.0005,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1510.48858,1.08478,1638.54719,0.98072,0.95584,0.99002,0.99414,0.99449,307.05141,298.31448,334.73368,276.04068,294.34834
0.0005,0.005,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1514.78772,1.04066,1576.3741,0.98072,1.00004,0.99002,0.99414,0.99449,331.36663,298.31448,314.7176,276.04068,294.34834
0.0005,0.005,0.005,0.008,0.013,0.0004,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1509.0855,1.03736,1565.45952,0.98072,1.00004,0.99002,0.99414,1.00221,319.54024,298.31448,314.7176,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1508.04863,1.0319,1556.15908,0.98072,1.00004,0.99002,0.99959,1.00221,312.22974,304.58812,314.7176,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1505.50142,1.03327,1555.59114,0.98072,1.00004,0.99002,0.99959,0.99643,322.66756,304.58812,301.73257,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1510.42152,1.03017,1555.98512,0.98072,1.00004,0.99002,0.99959,0.99953,322.66756,296.52318,314.7176,282.16484,294.34834
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1505.50142,1.03327,1555.59114,0.98072,1.00004,0.99002,0.99959,0.99643,322.66756,304.58812,301.73257,282.16484,294.34834


Gather the 100 best sets of coefficients into one DataFrame:

In [57]:
df_best_nudged = pl.DataFrame()

for d, df in dict_nudged_results.items():
    df_here = df.with_columns(pl.lit(d).alias('dir'))
    df_best_nudged = pl.concat((df_best_nudged, df_here.filter(df_here['fitness'] == df_here['fitness'].min())[0]))

Save a copy:

In [58]:
df_best_nudged.write_csv(os.path.join('outputs', 'nudge_walk_best.csv'))

Gather the full history of changes for all sets into one DataFrame:

In [59]:
df_nudge_history = pl.DataFrame()

for key, df in dict_nudged_results.items():
    df_here = df.with_columns(pl.lit(key).alias('dir'))
    df_nudge_history = pl.concat((df_nudge_history, df_here))

In [60]:
df_nudge_history

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_err_rats,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_err_rats_q00,sum_err_rats_q02,sum_err_rats_q04,sum_err_rats_q06,sum_err_rats_q08,dir
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1518.12708,1.16375,1766.72794,0.93525,1.03912,1.05265,0.99959,1.00682,309.48511,304.58812,334.73368,274.97184,294.34834,"""randomseed00"""
0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1527.02075,1.07561,1642.47432,0.98072,1.03912,0.99002,0.99959,1.00682,309.48511,304.58812,334.73368,283.86551,294.34834,"""randomseed00"""
0.0005,0.004,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1530.73297,1.05384,1613.14189,0.98072,1.01735,0.99002,0.99959,1.00682,321.02216,304.58812,334.73368,276.04068,294.34834,"""randomseed00"""
0.0005,0.005,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1524.76896,1.03652,1580.45626,0.98072,1.00004,0.99002,0.99959,1.00682,335.07424,304.58812,314.7176,276.04068,294.34834,"""randomseed00"""
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1508.04863,1.0319,1556.15908,0.98072,1.00004,0.99002,0.99959,1.00221,312.22974,304.58812,314.7176,282.16484,294.34834,"""randomseed00"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,1512.36229,1.0352,1565.6017,0.98072,1.00004,0.99002,0.99959,0.99449,322.66756,304.58812,314.7176,276.04068,294.34834,"""randomseed99"""
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1505.50142,1.03327,1555.59114,0.98072,1.00004,0.99002,0.99959,0.99643,322.66756,304.58812,301.73257,282.16484,294.34834,"""randomseed99"""
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,1510.42152,1.03017,1555.98512,0.98072,1.00004,0.99002,0.99959,0.99953,322.66756,296.52318,314.7176,282.16484,294.34834,"""randomseed99"""


Save a copy:

In [61]:
df_nudge_history.write_csv(os.path.join('outputs', 'nudge_walk_history.csv'))